# FMCG Global Demand Planning and Forecasting

## Notebook 08 – Model Training

### Objective

This notebook trains and compares multiple regression models for forecasting FMCG demand.

The models are evaluated using:

- MAE
- RMSE
- R²
- MAPE

The best-performing model will be used for explainability and deployment.

In [1]:
import os
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
X_train = pd.read_csv("data/processed/X_train.csv")
X_test = pd.read_csv("data/processed/X_test.csv")

y_train = pd.read_csv("data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("data/processed/y_test.csv").squeeze()

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


In [3]:
print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)

print()

print("Training Target   :", y_train.shape)
print("Testing Target    :", y_test.shape)

Training Features : (807640, 26)
Testing Features  : (201910, 26)

Training Target   : (807640,)
Testing Target    : (201910,)


In [4]:
def evaluate_model(model, X_test, y_test):

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    mape = mean_absolute_percentage_error(y_test, predictions)

    r2 = r2_score(y_test, predictions)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2
    }

In [5]:
baseline_prediction = np.repeat(y_train.mean(), len(y_test))

baseline_results = {

    "MAE": mean_absolute_error(y_test, baseline_prediction),

    "RMSE": np.sqrt(mean_squared_error(y_test, baseline_prediction)),

    "MAPE": mean_absolute_percentage_error(y_test, baseline_prediction),

    "R2": r2_score(y_test, baseline_prediction)

}

baseline_results

{'MAE': 34.522985970044225,
 'RMSE': np.float64(42.23286564328594),
 'MAPE': 892827850134326.8,
 'R2': -0.02770857144600103}

In [6]:
lr = LinearRegression()

lr.fit(X_train, y_train)

lr_results = evaluate_model(
    lr,
    X_test,
    y_test
)

lr_results

{'MAE': 15.692197954875402,
 'RMSE': np.float64(23.333847304839004),
 'MAPE': 895876838751720.8,
 'R2': 0.686280452270539}

In [7]:
rf = RandomForestRegressor(

    n_estimators=200,

    random_state=42,

    n_jobs=-1

)

rf.fit(
    X_train,
    y_train
)

rf_results = evaluate_model(
    rf,
    X_test,
    y_test
)

rf_results

{'MAE': 0.030605987816353743,
 'RMSE': np.float64(0.5691203915110391),
 'MAPE': 30334780314.11995,
 'R2': 0.9998133718416935}

In [8]:
gb = GradientBoostingRegressor(

    random_state=42

)

gb.fit(
    X_train,
    y_train
)

gb_results = evaluate_model(
    gb,
    X_test,
    y_test
)

gb_results

{'MAE': 1.4756300917782093,
 'RMSE': np.float64(2.329051398063914),
 'MAPE': 53615788494685.5,
 'R2': 0.996874445884285}

In [9]:
results = pd.DataFrame({

    "Model": [

        "Baseline",

        "Linear Regression",

        "Random Forest",

        "Gradient Boosting"

    ],

    "MAE": [

        baseline_results["MAE"],

        lr_results["MAE"],

        rf_results["MAE"],

        gb_results["MAE"]

    ],

    "RMSE": [

        baseline_results["RMSE"],

        lr_results["RMSE"],

        rf_results["RMSE"],

        gb_results["RMSE"]

    ],

    "MAPE": [

        baseline_results["MAPE"],

        lr_results["MAPE"],

        rf_results["MAPE"],

        gb_results["MAPE"]

    ],

    "R2": [

        baseline_results["R2"],

        lr_results["R2"],

        rf_results["R2"],

        gb_results["R2"]

    ]

})

results = results.sort_values("RMSE").reset_index(drop=True)

results

,Model,MAE,RMSE,MAPE,R2
0,Random Forest,0.030606,0.569120,3.033478e+10,0.999813
1,Gradient Boosting,1.475630,2.329051,5.361579e+13,0.996874
2,Linear Regression,15.692198,23.333847,8.958768e+14,0.686280
3,Baseline,34.522986,42.232866,8.928279e+14,-0.027709


In [10]:
best_model = rf

print("Best Model Selected: Random Forest")

Best Model Selected: Random Forest


In [11]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    best_model,
    "models/fmcg_forecasting_model.pkl"
)

print("Model Saved Successfully")

Model Saved Successfully


In [12]:
os.makedirs("reports", exist_ok=True)

results.to_csv(
    "reports/model_results.csv",
    index=False
)

print("Results Saved Successfully")

Results Saved Successfully


In [13]:
results

,Model,MAE,RMSE,MAPE,R2
0,Random Forest,0.030606,0.569120,3.033478e+10,0.999813
1,Gradient Boosting,1.475630,2.329051,5.361579e+13,0.996874
2,Linear Regression,15.692198,23.333847,8.958768e+14,0.686280
3,Baseline,34.522986,42.232866,8.928279e+14,-0.027709


In [16]:
(y_test == 0).sum()

np.int64(659)

In [17]:
y_test.describe()

count    201910.000000
mean         53.806161
std          41.659748
min           0.000000
25%          22.000000
50%          42.000000
75%          78.000000
max         500.000000
Name: units_sold, dtype: float64

In [19]:
X_train.columns.tolist()

['lag_1',
 'lag_7',
 'lag_30',
 'lag_90',
 'rolling_mean_7',
 'rolling_mean_30',
 'rolling_std_7',
 'effective_price',
 'discount_pct',
 'promotion',
 'stock_on_hand',
 'inventory_cover',
 'lead_time_days',
 'temperature',
 'rain_mm',
 'is_weekend',
 'is_holiday',
 'month',
 'week',
 'quarter',
 'country_enc',
 'city_enc',
 'channel_enc',
 'category_enc',
 'brand_enc',
 'sku_enc']

In [20]:
def mape_without_zeros(y_true, y_pred):

    mask = y_true != 0

    return np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask]) / y_true[mask]
        )
    ) * 100